<a href="https://colab.research.google.com/github/sweet-cross/cross_notebooks/blob/main/notebooks/workshop_20260309_assumptions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
if 'google.colab' in sys.modules:
    # Installs the repo as a package, along with its dependencies
    !pip install git+https://github.com/sweet-cross/cross_notebooks.git

# CROSS Plots

This notebook allows you to replicate the plots presenting the CROSS results.

## Packages and options

In [18]:
from crosscontract import CrossClient
import getpass
import pandas as pd
from typing import Self

## Connecting to CROSS

In [11]:
username = input("Enter your username or mail address: ")
password = getpass.getpass("Enter your password: ")
my_client = CrossClient(username=username, password=password)

In [15]:
cr = my_client.contracts.get("result_storage_output")
cr.contract.title

'Result submission - Storage output'

In [ ]:
from crosscontract.crossclient.services import ContractResource
class ResultVariable:

    def __init__(
        self,
        contract_resource: ContractResource,
        label: str | None = None,
        unit: str | None = None,
    ):
        self._contract_resource = contract_resource
        self.contract_name = contract_resource.contract.name
        self.label = label or self._clean_title(contract_resource.contract.title)
        self.unit = unit

    def __repr__(self):
        return (f"ResultVariable(contract_name='{self.contract_name}', " 
                f"label='{self.label}', unit='{self.unit}')")
    @property
    def contract_resource(self):
        """ContractResource as obtained from CROSS platform"""
        return self._contract_resource

    @classmethod
    def from_client(
        cls,
        client: CrossClient,
        contract_name: str,
        label: str | None = None,
        **kwargs,
    ) -> Self:
        """Build from a contract fetched via the client.

        Label is derived from the contract title unless explicitly overridden.
        
        Args:
            client: An instance of CrossClient to fetch contract details.
            contract_name: The name of the contract to fetch.
        """
        cr = client.contracts.get(contract_name)
        return cls(contract_resource=cr, label=label, **kwargs)

    @property
    def description(self) -> str:
        """A human-readable description of the variable"""
        return self.contract_resource.contract.description

    @property
    def title(self) -> str:
        """A human-readable title of the variable"""
        return self.contract_resource.contract.title

    @staticmethod
    def _clean_title(title: str) -> str:
        """Strip common prefixes like 'Result submission - ' from contract titles.
        
        Args:
            title: The original contract title.
        Returns:
            The cleaned title with known prefixes removed."""
        prefixes = ["Result submission - ", "Result Submission - "]
        for prefix in prefixes:
            if title.startswith(prefix):
                return title[len(prefix):]
        return title

    def get_data(
        self,
        scenario_group: str = "cross202506",
        scenario_name: str | None = None,
        scenario_variant: str | None = None,
        model: str | None = None,
    ) -> pd.DataFrame:
        """Get CROSS 2025 data for a specified contract.

        Args:
            scenario_group (str, optional): The name of the scenario group to filter results by.
                Defaults to "cross202506".
            scenario_name (str | None, optional): The name of the scenario to filter results by.
                Defaults to None, which means no filtering by scenario name.
            scenario_variant (str | None, optional): The name of the scenario variant to filter results by.
                Defaults to None, which means no filtering by scenario variant.
            model (str | None, optional): The name of the model to filter results by.
                Defaults to None, which means no filtering by model.
                Only valid in case of a result contract.

        Returns:
            pd.DataFrame: A DataFrame containing the results for the specified contract.
        """
        filters = {"scenario_group": scenario_group}
        if scenario_name is not None:
            filters["scenario_name"] = scenario_name
        if scenario_variant is not None:
            filters["scenario_variant"] = scenario_variant
        if self.contract_name.startswith("result_") and model is not None:
            filters["model"] = model
        df = (
            self.contract_resource.get_data(filters=filters)
            .drop(columns=["scenario_group"])
        )
        return df


def build_registry(client) -> dict[str, ResultVariable]:
    contracts = [
        ("result_electricity_consumption", {}),
        ("result_electricity_supply", {}),
        ("result_h2_fec", {"label": "H₂ Final Energy Consumption"}),
        ("result_h2_supply", {"label": "H₂ Supply"}),
        ("result_liquids_consumption", {}),
        ("result_liquids_supply", {}),
        ("result_methane_consumption", {}),
        ("result_methane_supply", {}),
        ("result_process_heat_energy_production", {}),
        ("result_space_heat_energy_supply", {}),
        ("result_district_heat_energy_production", {}),
        ("result_passenger_road_public_fec", {}),
        ("result_passenger_road_private_fec", {}),
        ("result_freight_road_fec", {}),
        ("result_storage_installed_volume", {}),
        ("result_storage_output", {}),
        ("result_elec_cons_typical_day", {}),
        ("result_elec_supply_typical_day", {}),
    ]
    return {
        "_".join(name.split("_")[1:]): ResultVariable.from_client(client, name, **overrides)
        for name, overrides in contracts
    }

variable_registry = build_registry(my_client)

In [24]:
variable_registry["electricity_consumption"]